# Задача 1. Tracer

Реализовать декоратор, который показывает стек вызовов рекурсивных функций.

На каждом шаге должны быть видны:

- вход в функцию;
- отступ, соответствующий глубине вызова;
- возврат из функции с результатом.

## Объяснение решения

Декоратор добавляет действия до и после вызова функции.

Перед вызовом печатаем имя функции и аргументы. Для вложенного вызова
увеличиваем глубину на 1. Один уровень глубины — два пробела отступа.

После выполнения печатаем результат и возвращаем его без изменения.
В `finally` восстанавливаем прежнюю глубину, в том числе после ошибки.
Поэтому каждый вход и соответствующий выход имеют одинаковый отступ.

`_depth` хранит общую глубину. `ContextVar` позволяет сохранить старое
значение и затем восстановить его. `wraps` сохраняет имя и описание функции.

`->` означает вход, `<-` — возврат, `!!` — исключение.
`enabled=False` отключает вывод, а через `stream` можно записать его в файл.

In [1]:
from contextvars import ContextVar
from functools import wraps
import sys


# общая глубина для всех функций с декоратором
_depth = ContextVar("tracer_depth", default=0)


def _print_call(output, depth, name, args, kwargs):
    """Строки существуют только во время печати, а не до конца рекурсии."""
    # собираем позиционные и именованные аргументы для вывода
    arguments = [repr(value) for value in args]
    arguments.extend(f"{key}={value!r}" for key, value in kwargs.items())
    print(f"{'  ' * depth}-> {name}({', '.join(arguments)})", file=output)


def _print_return(output, depth, name, result):
    print(f"{'  ' * depth}<- {name} = {result!r}", file=output)


def _print_exception(output, depth, name, error):
    print(
        f"{'  ' * depth}!! {name} raised {type(error).__name__}: {error}",
        file=output,
    )


def tracer(func=None, *, enabled=True, stream=None):
    """Показать входы, возвраты и исключения с отступом по глубине вызова.

    Поддерживаются обе формы: ``@tracer`` и ``@tracer(stream=...)``.
    ``enabled=False`` возвращает исходную функцию без трассировки.
    При ``stream=None`` текущий ``sys.stdout`` выбирается при каждом вызове.
    Декоратор предназначен для обычных синхронных функций.
    """

    def decorate(function):
        # если трассировка выключена, возвращаем функцию без обёртки
        if not enabled:
            return function

        # сохраняем имя и описание исходной функции
        @wraps(function)
        def wrapper(*args, **kwargs):
            output = sys.stdout if stream is None else stream
            depth = _depth.get()

            # показываем вход и увеличиваем глубину для вложенных вызовов
            _print_call(output, depth, function.__name__, args, kwargs)
            token = _depth.set(depth + 1)
            try:
                result = function(*args, **kwargs)
            except BaseException as error:
                # показываем ошибку и передаём её дальше
                _print_exception(output, depth, function.__name__, error)
                raise
            else:
                # показываем результат и возвращаем его без изменения
                _print_return(output, depth, function.__name__, result)
                return result
            finally:
                # восстанавливаем глубину даже после ошибки
                _depth.reset(token)

        return wrapper

    # поддерживаем запись и @tracer, и @tracer(...)
    if func is None:
        return decorate
    return decorate(func)

## Пример использования

Факториал 3 равен `3 × 2 × 1 = 6`.
Вызов `factorial(3)` ждёт результат `factorial(2)`,
а тот — результат `factorial(1)`.

In [2]:
@tracer
def factorial(n):
    # условие остановки рекурсии
    if n <= 1:
        return 1

    return n * factorial(n - 1)


print('Результат:', factorial(3))

-> factorial(3)
  -> factorial(2)
    -> factorial(1)
    <- factorial = 1
  <- factorial = 2
<- factorial = 6
Результат: 6


## Визуализация работы алгоритма

Рассмотрим ветвящуюся рекурсию `fibonacci(4)`.
Каждый вызов, кроме базовых случаев 0 и 1, вызывает функцию дважды.

```text
fibonacci(4) → 3
├── fibonacci(3) → 2
│   ├── fibonacci(2) → 1
│   │   ├── fibonacci(1) → 1
│   │   └── fibonacci(0) → 0
│   └── fibonacci(1) → 1
└── fibonacci(2) → 1
    ├── fibonacci(1) → 1
    └── fibonacci(0) → 0
```

Сначала полностью вычисляется левая ветвь, затем правая.
В каждой ветви самый глубокий вызов возвращает результат первым,
после чего продолжаются ожидающие внешние вызовы.
Результаты двух дочерних вызовов складываются.

In [3]:
@tracer
def fibonacci(n):
    if n <= 1:
        return n

    return fibonacci(n - 1) + fibonacci(n - 2)


print('Результат:', fibonacci(4))

-> fibonacci(4)
  -> fibonacci(3)
    -> fibonacci(2)
      -> fibonacci(1)
      <- fibonacci = 1
      -> fibonacci(0)
      <- fibonacci = 0
    <- fibonacci = 1
    -> fibonacci(1)
    <- fibonacci = 1
  <- fibonacci = 2
  -> fibonacci(2)
    -> fibonacci(1)
    <- fibonacci = 1
    -> fibonacci(0)
    <- fibonacci = 0
  <- fibonacci = 1
<- fibonacci = 3
Результат: 3


## Оценка сложности

Пусть `T` — время исходного алгоритма, `h` — максимальная глубина,
`L` — количество напечатанных символов.

**Время с трассировкой: `O(T + L)`**

Для каждого вызова печатаем вход и выход либо ошибку.
Считаем, что форматирование и запись занимают время, пропорциональное
длине текста. Отступы тоже учитываются: для цепочки глубины `h`
только они могут занять `O(h²)` символов.

**Дополнительная память декоратора: `O(h + s)`**

`h` активных обёрток хранят служебные данные, `s` — размер самой длинной
формируемой строки. Готовые строки не хранятся до конца рекурсии.
Память исходной функции учитывается отдельно. Если сохранять весь вывод
в `StringIO`, дополнительно требуется `O(L)` памяти.

## Тесты

Проверяем обычную, ветвящуюся и взаимную рекурсию, аргументы,
возврат `None`, отключение вывода и восстановление глубины после ошибки.

In [4]:
from contextlib import redirect_stdout
import inspect
import io
import unittest



class TestTracer(unittest.TestCase):
    def test_factorial_result_and_exact_stack(self):
        output = io.StringIO()

        @tracer(stream=output)
        def factorial(n):
            if n <= 1:
                return 1
            return n * factorial(n - 1)

        self.assertEqual(factorial(3), 6)
        self.assertEqual(
            output.getvalue().splitlines(),
            [
                "-> factorial(3)",
                "  -> factorial(2)",
                "    -> factorial(1)",
                "    <- factorial = 1",
                "  <- factorial = 2",
                "<- factorial = 6",
            ],
        )

    def test_branching_recursion(self):
        output = io.StringIO()

        @tracer(stream=output)
        def fibonacci(n):
            if n <= 1:
                return n
            return fibonacci(n - 1) + fibonacci(n - 2)

        self.assertEqual(fibonacci(4), 3)
        self.assertEqual(
            output.getvalue().splitlines(),
            [
                "-> fibonacci(4)",
                "  -> fibonacci(3)",
                "    -> fibonacci(2)",
                "      -> fibonacci(1)",
                "      <- fibonacci = 1",
                "      -> fibonacci(0)",
                "      <- fibonacci = 0",
                "    <- fibonacci = 1",
                "    -> fibonacci(1)",
                "    <- fibonacci = 1",
                "  <- fibonacci = 2",
                "  -> fibonacci(2)",
                "    -> fibonacci(1)",
                "    <- fibonacci = 1",
                "    -> fibonacci(0)",
                "    <- fibonacci = 0",
                "  <- fibonacci = 1",
                "<- fibonacci = 3",
            ],
        )

    def test_kwargs_and_return_value(self):
        output = io.StringIO()

        @tracer(stream=output)
        def calculate(a, b=0, *, scale=1):
            return (a + b) * scale

        self.assertEqual(calculate(2, b=3, scale=4), 20)
        self.assertEqual(
            output.getvalue(),
            "-> calculate(2, b=3, scale=4)\n<- calculate = 20\n",
        )

    def test_metadata_and_signature_are_preserved(self):
        @tracer(enabled=False)
        def original(a: int, b: int = 2) -> int:
            """Документация исходной функции."""
            return a + b

        decorated = tracer(original, stream=io.StringIO())
        self.assertEqual(decorated.__name__, original.__name__)
        self.assertEqual(decorated.__doc__, original.__doc__)
        self.assertEqual(decorated.__annotations__, original.__annotations__)
        self.assertEqual(inspect.signature(decorated), inspect.signature(original))
        self.assertIs(decorated.__wrapped__, original)

    def test_disabled_tracer_does_not_print_or_change_depth(self):
        output = io.StringIO()

        @tracer(enabled=False, stream=output)
        def silent(n):
            return n if n == 0 else silent(n - 1) + 1

        self.assertEqual(silent(3), 3)
        self.assertEqual(output.getvalue(), "")

        @tracer(stream=output)
        def visible(value):
            return value

        @tracer(enabled=False)
        def silent_parent(value):
            return visible(value)

        self.assertEqual(silent_parent(4), 4)
        self.assertEqual(output.getvalue(), "-> visible(4)\n<- visible = 4\n")

    def test_exception_is_reraised_and_depth_is_restored(self):
        output = io.StringIO()
        problem = ValueError("boom")

        @tracer(stream=output)
        def fail(n):
            if n == 0:
                raise problem
            return fail(n - 1)

        with self.assertRaises(ValueError) as caught:
            fail(2)
        self.assertIs(caught.exception, problem)

        @tracer(stream=output)
        def recover():
            return 7

        self.assertEqual(recover(), 7)
        self.assertEqual(
            output.getvalue().splitlines(),
            [
                "-> fail(2)",
                "  -> fail(1)",
                "    -> fail(0)",
                "    !! fail raised ValueError: boom",
                "  !! fail raised ValueError: boom",
                "!! fail raised ValueError: boom",
                "-> recover()",
                "<- recover = 7",
            ],
        )

    def test_sequential_calls_start_at_zero_depth(self):
        output = io.StringIO()

        @tracer(stream=output)
        def identity(value):
            return value

        self.assertEqual(identity(1), 1)
        self.assertEqual(identity(2), 2)
        self.assertEqual(
            output.getvalue(),
            "-> identity(1)\n<- identity = 1\n"
            "-> identity(2)\n<- identity = 2\n",
        )

    def test_mutual_recursion_shares_depth(self):
        output = io.StringIO()

        @tracer(stream=output)
        def is_even(n):
            return True if n == 0 else is_odd(n - 1)

        @tracer(stream=output)
        def is_odd(n):
            return False if n == 0 else is_even(n - 1)

        self.assertFalse(is_even(3))
        self.assertEqual(
            output.getvalue().splitlines(),
            [
                "-> is_even(3)",
                "  -> is_odd(2)",
                "    -> is_even(1)",
                "      -> is_odd(0)",
                "      <- is_odd = False",
                "    <- is_even = False",
                "  <- is_odd = False",
                "<- is_even = False",
            ],
        )

    def test_default_stream_is_selected_when_called(self):
        @tracer
        def greet(name):
            return f"Hello, {name}!"

        first_output = io.StringIO()
        second_output = io.StringIO()
        with redirect_stdout(first_output):
            self.assertEqual(greet("Ada"), "Hello, Ada!")
        with redirect_stdout(second_output):
            self.assertEqual(greet("Bob"), "Hello, Bob!")

        self.assertEqual(
            first_output.getvalue(),
            "-> greet('Ada')\n<- greet = 'Hello, Ada!'\n",
        )
        self.assertEqual(
            second_output.getvalue(),
            "-> greet('Bob')\n<- greet = 'Hello, Bob!'\n",
        )

    def test_empty_arguments_and_none_result(self):
        output = io.StringIO()

        @tracer(stream=output)
        def do_nothing():
            return None

        self.assertIsNone(do_nothing())
        self.assertEqual(
            output.getvalue(), "-> do_nothing()\n<- do_nothing = None\n"
        )


import sys

suite = unittest.TestLoader().loadTestsFromTestCase(TestTracer)
runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful()

test_branching_recursion (__main__.TestTracer.test_branching_recursion) ... 

ok


test_default_stream_is_selected_when_called (__main__.TestTracer.test_default_stream_is_selected_when_called) ... 

ok


test_disabled_tracer_does_not_print_or_change_depth (__main__.TestTracer.test_disabled_tracer_does_not_print_or_change_depth) ... 

ok


... (11 outputs left out)

ok


test_sequential_calls_start_at_zero_depth (__main__.TestTracer.test_sequential_calls_start_at_zero_depth) ... 

ok


----------------------------------------------------------------------
Ran 10 tests in 0.003s

OK
